In [ ]:
# ==== Hand gesture classifiers: SAM training + MCC evaluation + ONNX export (single cell) ====
# Retrains both classifiers used by app.py on the repo's CSVs:
#   keypoint_classifier      : 42 wrist-relative landmark coords -> {Open, Close, Pointer}
#   point_history_classifier : 16 fingertip (x, y) steps (LSTM) -> {Stop, Clockwise, Counter Clockwise, Move}
# Optimiser: SAM (Foret et al., 2021) wrapping Adam. Model selection: validation MCC.
# Reporting: test-set MCC (primary), confusion matrix, accuracy / macro-F1 for reference.
# Export: ONNX with the same input/output names app.py expects, written as *_sam.onnx.
import os, sys, subprocess, json, random, time

def _pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

try:
    import onnx, onnxruntime  # noqa: F401
except ImportError:
    _pip('onnx', 'onnxruntime')
import numpy as np
import torch, torch.nn as nn
import onnxruntime as ort
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, confusion_matrix, f1_score, accuracy_score

REPO_URL = 'https://github.com/dhaneshkk/hand-gesture-recognition-using-onnx'
BRANCH = os.environ.get('HGR_BRANCH', 'main')
REPO = os.environ.get('HGR_REPO_DIR', '')
if not REPO:
    if os.path.isfile('app.py') and os.path.isdir('model'):
        REPO = os.getcwd()
    else:
        REPO = '/content/hand-gesture-recognition-using-onnx'
        if not os.path.isdir(REPO):
            subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, REPO], check=True)

SEED = int(os.environ.get('HGR_SEED', 42))
MAX_EPOCHS = int(os.environ.get('HGR_EPOCHS', 300))
PATIENCE = 50
SAM_RHO = 0.05
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f'repo={REPO} device={DEVICE} seed={SEED}')

# ---------------------------------------------------------------- data
def load_csv(path, n_features):
    """label + n_features columns. Rows logged by the old app.py carried an extra
    trackid column (label, trackid, features...); detect and drop it."""
    rows = [l.strip().split(',') for l in open(path) if l.strip()]
    X, y, dropped = [], [], 0
    for r in rows:
        if len(r) == n_features + 1:
            feats = r[1:]
        elif len(r) == n_features + 2:
            feats = r[2:]
        else:
            dropped += 1
            continue
        y.append(int(float(r[0]))); X.append([float(v) for v in feats])
    if dropped:
        print(f'  {os.path.basename(path)}: skipped {dropped} malformed rows')
    return np.asarray(X, np.float32), np.asarray(y, np.int64)

def split(X, y):
    """Stratified 70 / 15 / 15. The test split is never used for early stopping."""
    Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
    Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.50, stratify=ytmp, random_state=SEED)
    return (Xtr, ytr), (Xva, yva), (Xte, yte)

# ---------------------------------------------------------------- SAM
class SAM(torch.optim.Optimizer):
    """Sharpness-Aware Minimisation: min_w max_{||e||<=rho} L(w + e).
    first_step: w <- w + rho * g / ||g||   second_step: restore w, step base optimiser
    using the gradient taken at the perturbed point."""
    def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
        assert rho >= 0
        super().__init__(params, dict(rho=rho, **kwargs))
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups

    @torch.no_grad()
    def _grad_norm(self):
        dev = self.param_groups[0]['params'][0].device
        norms = [p.grad.norm(2).to(dev) for g in self.param_groups for p in g['params'] if p.grad is not None]
        return torch.norm(torch.stack(norms), 2) if norms else torch.tensor(0.0, device=dev)

    @torch.no_grad()
    def first_step(self):
        scale_base = self._grad_norm() + 1e-12
        for g in self.param_groups:
            scale = g['rho'] / scale_base
            for p in g['params']:
                if p.grad is None:
                    continue
                e = p.grad * scale
                p.add_(e)
                self.state[p]['e'] = e
        self.zero_grad()

    @torch.no_grad()
    def second_step(self):
        for g in self.param_groups:
            for p in g['params']:
                if 'e' in self.state[p]:
                    p.sub_(self.state[p].pop('e'))
        self.base_optimizer.step()
        self.zero_grad()

    def step(self, closure=None):
        raise RuntimeError('use first_step / second_step')

# ---------------------------------------------------------------- models (mirror the shipped architectures)
class KeypointMLP(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(0.2), nn.Linear(42, 20), nn.ReLU(),
            nn.Dropout(0.4), nn.Linear(20, 10), nn.ReLU(),
            nn.Linear(10, n_classes))
    def forward(self, x):
        return self.net(x)

class HistoryLSTM(nn.Module):
    def __init__(self, n_classes, steps=16, dim=2):
        super().__init__()
        self.steps, self.dim = steps, dim
        self.drop = nn.Dropout(0.2)
        self.lstm = nn.LSTM(dim, 16, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(16, 10), nn.ReLU(), nn.Linear(10, n_classes))
    def forward(self, x):
        x = self.drop(x.reshape(-1, self.steps, self.dim))
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

# ---------------------------------------------------------------- train / eval
def mcc_multiclass(y_true, y_pred, n_classes):
    """Gorodkin's R_K multiclass MCC from the confusion matrix (cross-checked against sklearn)."""
    C = np.zeros((n_classes, n_classes), np.float64)
    for t, p in zip(y_true, y_pred):
        C[t, p] += 1
    t_k, p_k, c, s = C.sum(1), C.sum(0), np.trace(C), C.sum()
    denom = np.sqrt((s * s - (p_k ** 2).sum()) * (s * s - (t_k ** 2).sum()))
    return float((c * s - (t_k * p_k).sum()) / denom) if denom > 0 else 0.0

@torch.no_grad()
def predict(model, X):
    model.eval()
    return model(torch.from_numpy(X).to(DEVICE)).argmax(1).cpu().numpy()

def train(model, data, n_classes, name, batch_size=128, lr=1e-3):
    (Xtr, ytr), (Xva, yva), _ = data
    model.to(DEVICE)
    opt = SAM(model.parameters(), torch.optim.Adam, rho=SAM_RHO, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    Xt, yt = torch.from_numpy(Xtr).to(DEVICE), torch.from_numpy(ytr).to(DEVICE)
    best_mcc, best_state, bad, t0 = -2.0, None, 0, time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(len(Xt), device=DEVICE)
        for i in range(0, len(Xt), batch_size):
            idx = perm[i:i + batch_size]
            xb, yb = Xt[idx], yt[idx]
            loss_fn(model(xb), yb).backward(); opt.first_step()   # ascent to w + e
            loss_fn(model(xb), yb).backward(); opt.second_step()  # descent with sharpness-aware grad
        va_mcc = mcc_multiclass(yva, predict(model, Xva), n_classes)
        if va_mcc > best_mcc + 1e-6:
            best_mcc, bad = va_mcc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if epoch % 25 == 0 or bad == 0 and epoch < 5:
            print(f'  [{name}] epoch {epoch:3d}  val MCC {va_mcc:.4f}  best {best_mcc:.4f}')
        if bad >= PATIENCE:
            break
    model.load_state_dict(best_state)
    print(f'  [{name}] stopped at epoch {epoch}, best val MCC {best_mcc:.4f} ({time.time()-t0:.1f}s)')
    return model

def report(name, y_true, y_pred, labels):
    n = len(labels)
    mcc = mcc_multiclass(y_true, y_pred, n)
    sk = matthews_corrcoef(y_true, y_pred)
    assert abs(mcc - sk) < 1e-9, (mcc, sk)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n)))
    print(f'\n{name}: test MCC = {mcc:.4f}   (accuracy {accuracy_score(y_true, y_pred):.4f}, '
          f'macro-F1 {f1_score(y_true, y_pred, average="macro"):.4f}, n={len(y_true)})')
    w = max(len(l) for l in labels)
    print(' ' * (w + 2) + '  '.join(f'{i:>5d}' for i in range(n)) + '   <- predicted')
    for i, row in enumerate(cm):
        print(f'{labels[i]:>{w}} {i}' + '  '.join(f'{v:>5d}' for v in row))
    return mcc

# ---------------------------------------------------------------- ONNX export (same interface as app.py)
class KeypointExport(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        return torch.argmax(self.m(x), dim=1)

class HistoryExport(nn.Module):
    """class_ids = argmax(softmax); if max prob < score_threshold -> 0 (Stop), as in make_argmax.py."""
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x, score_threshold):
        p = torch.softmax(self.m(x), dim=1)
        s, ids = torch.max(p, dim=1)
        return torch.where(s < score_threshold, torch.zeros_like(ids), ids)

def export(wrapper, args, input_names, path, dynamic_axes):
    wrapper.eval().cpu()
    kw = dict(opset_version=13, input_names=input_names, output_names=['class_ids'], dynamic_axes=dynamic_axes)
    try:
        torch.onnx.export(wrapper, args, path, dynamo=False, **kw)
    except TypeError:  # older torch without the dynamo kwarg
        torch.onnx.export(wrapper, args, path, **kw)
    onnx.checker.check_model(onnx.load(path))
    return path

def onnx_predict(path, X, score_threshold=None):
    s = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
    feed = {'input': X.astype(np.float32)}
    if score_threshold is not None:
        feed['score_threshold'] = np.asarray(score_threshold, np.float32)
    return s.run(None, feed)[0]

def labels_of(path):
    return [l.strip().lstrip('﻿') for l in open(path, encoding='utf-8') if l.strip()]

# ---------------------------------------------------------------- run
results = {}
KP_DIR = os.path.join(REPO, 'model/keypoint_classifier')
PH_DIR = os.path.join(REPO, 'model/point_history_classifier')

for task in ('keypoint', 'point_history'):
    if task == 'keypoint':
        d, n_feat, csv_name, label_csv = KP_DIR, 42, 'keypoint.csv', 'keypoint_classifier_label.csv'
        shipped, out_name = 'keypoint_classifier.onnx', 'keypoint_classifier_sam.onnx'
    else:
        d, n_feat, csv_name, label_csv = PH_DIR, 32, 'point_history.csv', 'point_history_classifier_label.csv'
        shipped, out_name = 'point_history_classifier_lstm.onnx', 'point_history_classifier_lstm_sam.onnx'
    labels = labels_of(os.path.join(d, label_csv))
    X, y = load_csv(os.path.join(d, csv_name), n_feat)
    n_classes = len(labels)
    assert y.max() < n_classes, f'label {y.max()} >= {n_classes} classes in {label_csv}'
    print(f'\n=== {task}: {len(y)} rows, class counts {np.bincount(y, minlength=n_classes).tolist()}')
    data = split(X, y)
    Xte, yte = data[2]

    torch.manual_seed(SEED)
    model = KeypointMLP(n_classes) if task == 'keypoint' else HistoryLSTM(n_classes)
    model = train(model, data, n_classes, task)
    mcc_torch = report(f'{task} (SAM, PyTorch)', yte, predict(model, Xte), labels)

    out = os.path.join(d, out_name)
    if task == 'keypoint':
        export(KeypointExport(model), (torch.zeros(1, 42),), ['input'], out,
               {'input': {0: 'batch'}, 'class_ids': {0: 'batch'}})
        y_onnx = onnx_predict(out, Xte)
        y_ship = onnx_predict(os.path.join(d, shipped), Xte)
    else:
        export(HistoryExport(model), (torch.zeros(1, 32), torch.tensor(0.0)), ['input', 'score_threshold'], out,
               {'input': {0: 'batch'}, 'class_ids': {0: 'batch'}})
        y_onnx = onnx_predict(out, Xte, 0.0)   # threshold 0 -> pure argmax, comparable to PyTorch
        y_ship = onnx_predict(os.path.join(d, shipped), Xte, 0.0)
    parity = float((y_onnx == predict(model, Xte)).mean())
    print(f'  ONNX/PyTorch prediction parity on test: {parity:.4f}  -> {out}')
    assert parity > 0.995
    mcc_ship = report(f'{task} (shipped model, same test rows*)', yte, y_ship, labels)
    results[task] = dict(test_mcc_sam=mcc_torch, test_mcc_shipped=mcc_ship, onnx_parity=parity,
                         onnx_path=os.path.relpath(out, REPO), n_test=int(len(yte)))

print('\n* The shipped models were trained on these CSVs with their own random split, so some of\n'
      '  these test rows were probably in their training set: treat their MCC as optimistic.')
print(json.dumps(results, indent=2))
print('\nUse the new models:  python app.py --keypoint_model model/keypoint_classifier/keypoint_classifier_sam.onnx '
      '--point_history_model model/point_history_classifier/point_history_classifier_lstm_sam.onnx')
